# ASX Instrument Data Pull — Interactive Brokers

Download historical bar data for ASX (Australian Securities Exchange) stocks
from Interactive Brokers TWS.

**Prerequisites:**
- TWS (Trader Workstation) or IB Gateway running locally
- API connections enabled in TWS (Edit > Global Configuration > API > Settings)
- Port 7496 (live) or 7497 (paper) open

**Timeframes pulled:**
- **1-minute bars** — 6 months lookback (IB max for intraday)
- **End-of-day (1-day) bars** — 10 years lookback

**Example instruments:** CBA, BHP, CSL, WES, NAB (top ASX stocks)

**Uses shared utilities from `data.ib_historical`** — rate limiting, chunking,
validation, retries, and catalog writing.

## 0. Check TWS Connection

In [ ]:
import sys
sys.path.insert(0, str(__import__("pathlib").Path.cwd().parents[1] / "packages" / "data"))

from data.ib_historical import check_tws_connection

TWS_HOST = "127.0.0.1"
TWS_PORT = 7496  # 7496 = live, 7497 = paper

if check_tws_connection(TWS_HOST, TWS_PORT):
    print(f"OK — TWS is accepting connections at {TWS_HOST}:{TWS_PORT}")
else:
    print(f"FAILED — Cannot connect to TWS at {TWS_HOST}:{TWS_PORT}")
    print()
    print("Please ensure:")
    print("  1. TWS is running and you are logged in")
    print("  2. API is enabled: Edit > Global Configuration > API > Settings")
    print(f"     - Socket port is set to {TWS_PORT}")
    print("  3. For paper trading, change TWS_PORT to 7497")

## 1. Configuration

In [ ]:
import datetime
import os
from pathlib import Path

from nautilus_trader.adapters.interactive_brokers.common import IBContract
from data.ib_historical import RateLimiter, connect_client, pull_bars, save_bars_to_catalog

# === TWS Settings ===
TWS_HOST = "127.0.0.1"
TWS_PORT = 7496
TWS_CLIENT_ID = 2  # Use a different client ID from other notebooks

# === Catalog ===
CATALOG_PATH = Path(
    os.environ.get(
        "NAUTILUS_STORE_PATH",
        str(Path.cwd().parents[2] / "backtest_catalog"),
    )
)

# === Shared rate limiter (one per session) ===
rate_limiter = RateLimiter()

print(f"Catalog: {CATALOG_PATH}")
print(f"Catalog exists: {CATALOG_PATH.exists()}")

## 2. Define ASX Instruments

Each instrument is an `IBContract` with:
- `secType="STK"` — equity/stock
- `exchange="ASX"` — Australian Securities Exchange
- `currency="AUD"` — Australian Dollar
- Price type: `LAST` (last traded price — appropriate for equities)

In [ ]:
# === ASX Instruments ===
# Add or remove instruments as needed.

ASX_INSTRUMENTS = {
    "CBA": IBContract(secType="STK", symbol="CBA", exchange="ASX", currency="AUD"),
    "BHP": IBContract(secType="STK", symbol="BHP", exchange="ASX", currency="AUD"),
    "CSL": IBContract(secType="STK", symbol="CSL", exchange="ASX", currency="AUD"),
    "WES": IBContract(secType="STK", symbol="WES", exchange="ASX", currency="AUD"),
    "NAB": IBContract(secType="STK", symbol="NAB", exchange="ASX", currency="AUD"),
}

# === Timeframe Configuration ===
PULLS = {
    "1-MINUTE-LAST": {
        "lookback": datetime.timedelta(days=180),   # ~6 months
        "use_rth": True,   # Regular trading hours only
        "timeout": 10,     # Seconds per chunk request
    },
    "1-DAY-LAST": {
        "lookback": datetime.timedelta(days=3650),  # ~10 years
        "use_rth": True,
        "timeout": 15,     # Daily bars can be slower to return
    },
}

print(f"Instruments: {list(ASX_INSTRUMENTS.keys())}")
for bar_spec, cfg in PULLS.items():
    print(f"  {bar_spec}: {cfg['lookback'].days} days lookback")

## 3. Connect to TWS

In [ ]:
client = await connect_client(
    host=TWS_HOST,
    port=TWS_PORT,
    client_id=TWS_CLIENT_ID,
)

## 4. Resolve Instruments

Request full instrument details from IB. This resolves tick sizes, lot sizes,
trading hours, margin requirements, etc.

In [ ]:
resolved_instruments = {}

for name, contract in ASX_INSTRUMENTS.items():
    try:
        instruments = await client.request_instruments(contracts=[contract])
        if instruments:
            resolved_instruments[name] = instruments[0]
            print(f"  {name}: {instruments[0].id} — resolved OK")
        else:
            print(f"  {name}: FAILED to resolve — no instrument returned")
    except Exception as e:
        print(f"  {name}: ERROR — {e}")

print(f"\nResolved {len(resolved_instruments)}/{len(ASX_INSTRUMENTS)} instruments")

if len(resolved_instruments) < len(ASX_INSTRUMENTS):
    missing = set(ASX_INSTRUMENTS) - set(resolved_instruments)
    print(f"WARNING: Missing instruments: {missing}")
    print("These will be skipped during the pull.")

## 5. Pull Historical Data

For each instrument and timeframe, this:
1. Chunks the date range into IB-compatible sizes
2. Rate-limits requests (max 55 per 10 minutes, 3s between requests)
3. Retries failed chunks up to 3 times with exponential backoff
4. Validates returned data (timestamps, prices)
5. Deduplicates overlapping chunks

**Estimated time:**
- 1-minute bars (6 months): ~180 chunks per instrument x 3s = ~9 min each
- 1-day bars (10 years): ~10 chunks per instrument x 3s = ~30s each
- Total for 5 instruments: ~50 minutes (with rate limit pauses)

In [ ]:
end = datetime.datetime.now(datetime.timezone.utc)
results = {}  # {(instrument_name, bar_spec): bars}

for name, contract in ASX_INSTRUMENTS.items():
    if name not in resolved_instruments:
        print(f"\nSkipping {name} — not resolved")
        continue

    for bar_spec, config in PULLS.items():
        start = end - config["lookback"]
        key = (name, bar_spec)

        print(f"\n{'=' * 60}")
        bars = await pull_bars(
            client=client,
            contract=contract,
            bar_spec=bar_spec,
            start=start,
            end=end,
            rate_limiter=rate_limiter,
            use_rth=config["use_rth"],
            timeout_per_chunk=config["timeout"],
            max_retries=3,
            retry_delay=5.0,
        )

        results[key] = bars

print(f"\n{'=' * 60}")
print(f"\nAll pulls complete. {len(results)} datasets:")
for (name, bar_spec), bars in results.items():
    print(f"  {name} {bar_spec}: {len(bars)} bars")

## 6. Save to Catalog

Write all pulled bars to the backtest catalog in NautilusTrader's streaming
format (Arrow IPC / feather files). Each instrument+timeframe gets its own
run directory.

The dashboard will automatically pick up these runs.

In [ ]:
for (name, bar_spec), bars in results.items():
    if not bars:
        print(f"  {name} {bar_spec}: no bars — skipping")
        continue

    print(f"\n{'=' * 60}")
    run_id = save_bars_to_catalog(bars, CATALOG_PATH)
    print(f"  View at: http://localhost:5173/runs/{run_id}")

print(f"\nAll data saved to {CATALOG_PATH}")
print("Run 'bun run dev' to view in the dashboard.")

## 7. Verify

Check that the saved data is readable from the catalog.

In [ ]:
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.model.data import Bar

catalog = ParquetDataCatalog(str(CATALOG_PATH))

# List backtest runs (most recent first)
runs = catalog.list_backtest_runs()
print(f"Total runs in catalog: {len(runs)}")

# Show the runs we just created
for (name, bar_spec), bars in results.items():
    if bars:
        bar_type_str = str(bars[0].bar_type)
        print(f"\n  {name} {bar_spec}:")
        print(f"    Bar type: {bar_type_str}")
        print(f"    Bars: {len(bars)}")
        print(f"    First: {datetime.datetime.fromtimestamp(bars[0].ts_event / 1e9, tz=datetime.timezone.utc)}")
        print(f"    Last:  {datetime.datetime.fromtimestamp(bars[-1].ts_event / 1e9, tz=datetime.timezone.utc)}")

## 8. Cleanup

In [ ]:
client._client.disconnect()
print("Disconnected from TWS")